# Face Recognition Attendance System

**Tools:** OpenCV (Computer Vision), NumPy, Pandas

**Pipeline:**
1. Setup folders
2. Capture face data (webcam)
3. Train LBPH face recognizer
4. Live recognition + attendance marking (CSV)
5. Generate attendance reports (Pandas)

> Run this notebook locally (not in a headless/cloud environment) since it needs webcam + GUI window access (`cv2.imshow`).


## 1. Imports & Config

In [2]:
!pip install opencv-contrib-python numpy pandas

   ---------------------------------------- 0.0/53.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/53.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/53.8 MB ? eta -:--:--
   ---------------------------------------- 0.5/53.8 MB 1.5 MB/s eta 0:00:35
    --------------------------------------- 1.0/53.8 MB 2.0 MB/s eta 0:00:27
   - -------------------------------------- 1.6/53.8 MB 2.2 MB/s eta 0:00:24
   - -------------------------------------- 2.1/53.8 MB 2.3 MB/s eta 0:00:23
   -- ------------------------------------- 3.1/53.8 MB 2.4 MB/s eta 0:00:22
   -- ------------------------------------- 3.9/53.8 MB 2.6 MB/s eta 0:00:20
   --- ------------------------------------ 4.5/53.8 MB 2.6 MB/s eta 0:00:19
   ---- ----------------------------------- 5.5/53.8 MB 2.9 MB/s eta 0:00:17
   ----- ---------------------------------- 6.8/53.8 MB 3.2 MB/s eta 0:00:15
   ------ --------------------------------- 8.1/53.8 MB 3.4 MB/s eta 0:00:14
   ------- ---------

In [5]:
import cv2, os
print(cv2.data.haarcascades)
path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
print("Exists:", os.path.exists(path))

C:\Users\faria\miniconda3\envs\yolovenv\lib\site-packages\cv2\data\
Exists: False


In [6]:
!pip list | findstr opencv

opencv-contrib-python     5.0.0.93


In [9]:
import cv2
import numpy as np
import pandas as pd
import os
import pickle
from datetime import datetime

# ---- Config ----
DATASET_DIR   = "dataset"          # raw face images, one subfolder per person
TRAINER_DIR   = "trainer"          # trained model + label map
ATTENDANCE_DIR = "attendance"      # daily CSV logs
MODEL_PATH    = os.path.join(TRAINER_DIR, "lbph_model.yml")
LABELS_PATH   = os.path.join(TRAINER_DIR, "labels.pkl")

FACE_SIZE = (200, 200)             # all face crops resized to this
SAMPLES_PER_PERSON = 60            # how many face images to capture per person
CONFIDENCE_THRESHOLD = 70          # LBPH: lower = more confident match (tune this)

for d in [DATASET_DIR, TRAINER_DIR, ATTENDANCE_DIR]:
    os.makedirs(d, exist_ok=True)

import urllib.request

os.makedirs("cascades", exist_ok=True)
cascade_path = "cascades/haarcascade_frontalface_default.xml"
if not os.path.exists(cascade_path):
    url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
    urllib.request.urlretrieve(url, cascade_path)

face_cascade = cv2.CascadeClassifier(cascade_path)
print("Setup complete. Cascade loaded:", not face_cascade.empty())


Setup complete. Cascade loaded: True


## 2. Capture Face Data

Run this once per person you want to register. It opens the webcam, detects your face,
and saves cropped grayscale face images into `dataset/<person_name>/`.

Press **q** to stop early.


In [15]:
import shutil

shutil.rmtree("dataset/Faria")
print("Deleted old samples for Faria.")

Deleted old samples for Faria.


In [16]:
def capture_faces(person_name, num_samples=SAMPLES_PER_PERSON):
    person_dir = os.path.join(DATASET_DIR, person_name)
    os.makedirs(person_dir, exist_ok=True)

    cap = cv2.VideoCapture(0)
    count = 0

    print(f"Capturing faces for '{person_name}'. Look at the camera...")
    while count < num_samples:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(80, 80))

        for (x, y, w, h) in faces:
            face_crop = gray[y:y+h, x:x+w]
            face_crop = cv2.resize(face_crop, FACE_SIZE)

            count += 1
            file_path = os.path.join(person_dir, f"{count}.jpg")
            cv2.imwrite(file_path, face_crop)

            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            cv2.putText(frame, f"Samples: {count}/{num_samples}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            break  # only take one face per frame

        cv2.imshow("Capturing Faces - press q to stop", frame)
        if cv2.waitKey(1) & 0xFF == ord('q') or count >= num_samples:
            break

    cap.release()
    cv2.destroyAllWindows()
    print(f"Done. Saved {count} images to {person_dir}")

# Example usage:
capture_faces("Faria")


Capturing faces for 'Faria'. Look at the camera...
Done. Saved 60 images to dataset\Faria


## 3. Train the Recognizer

Reads every image in `dataset/`, builds NumPy arrays of pixel data + integer labels,
and trains OpenCV's LBPH (Local Binary Patterns Histogram) face recognizer — a lightweight
algorithm well suited to small, self-collected datasets.


In [17]:
def train_recognizer():
    faces = []
    labels = []
    label_map = {}     # {id: person_name}
    current_id = 0

    for person_name in sorted(os.listdir(DATASET_DIR)):
        person_dir = os.path.join(DATASET_DIR, person_name)
        if not os.path.isdir(person_dir):
            continue

        label_map[current_id] = person_name

        for img_file in os.listdir(person_dir):
            img_path = os.path.join(person_dir, img_file)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, FACE_SIZE)
            faces.append(np.array(img, dtype=np.uint8))
            labels.append(current_id)

        current_id += 1

    if len(faces) == 0:
        print("No training data found. Run capture_faces() first.")
        return None, None

    labels = np.array(labels)

    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.train(faces, labels)
    recognizer.save(MODEL_PATH)

    with open(LABELS_PATH, "wb") as f:
        pickle.dump(label_map, f)

    print(f"Trained on {len(faces)} images across {len(label_map)} people.")
    print("Registered people:", list(label_map.values()))
    return recognizer, label_map

# Example usage:
recognizer, label_map = train_recognizer()


Trained on 60 images across 1 people.
Registered people: ['Faria']


In [19]:
import os
print(os.getcwd())

C:\Users\faria


## 4. Live Recognition + Attendance Marking

Opens the webcam, identifies faces using the trained model, and logs attendance
(name, date, time) to a CSV in `attendance/`. Each person is marked **once per day**.


In [20]:
def get_today_csv():
    today = datetime.now().strftime("%Y-%m-%d")
    return os.path.join(ATTENDANCE_DIR, f"attendance_{today}.csv")

def mark_attendance(name):
    csv_path = get_today_csv()

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
    else:
        df = pd.DataFrame(columns=["Name", "Date", "Time"])

    if name in df["Name"].values:
        return False  # already marked today

    now = datetime.now()
    new_row = pd.DataFrame([{
        "Name": name,
        "Date": now.strftime("%Y-%m-%d"),
        "Time": now.strftime("%H:%M:%S")
    }])
    df = pd.concat([df, new_row], ignore_index=True)
    df.to_csv(csv_path, index=False)
    return True


def run_attendance_system():
    if not os.path.exists(MODEL_PATH):
        print("No trained model found. Run train_recognizer() first.")
        return

    recognizer = cv2.face.LBPHFaceRecognizer_create()
    recognizer.read(MODEL_PATH)
    with open(LABELS_PATH, "rb") as f:
        label_map = pickle.load(f)

    cap = cv2.VideoCapture(0)
    marked_this_session = set()

    print("Starting attendance system. Press q to quit.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(80, 80))

        for (x, y, w, h) in faces:
            face_crop = cv2.resize(gray[y:y+h, x:x+w], FACE_SIZE)
            label_id, confidence = recognizer.predict(face_crop)

            if confidence < CONFIDENCE_THRESHOLD:
                name = label_map.get(label_id, "Unknown")
                color = (0, 255, 0)

                if name not in marked_this_session:
                    if mark_attendance(name):
                        print(f"Marked present: {name} at {datetime.now().strftime('%H:%M:%S')}")
                    marked_this_session.add(name)
            else:
                name = "Unknown"
                color = (0, 0, 255)

            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
            cv2.putText(frame, f"{name} ({int(confidence)})", (x, y-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        cv2.imshow("Attendance System - press q to quit", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Example usage:
run_attendance_system()


Starting attendance system. Press q to quit.
Marked present: Faria at 11:28:18


## 5. Attendance Reports (Pandas)

Combines all daily CSVs into one dataset and generates useful summaries:
- Full attendance log
- Attendance count per person
- Attendance percentage per person (out of total days recorded)
- A single day's report


In [23]:
def load_all_attendance():
    all_files = [os.path.join(ATTENDANCE_DIR, f) for f in os.listdir(ATTENDANCE_DIR) if f.endswith(".csv")]
    if not all_files:
        print("No attendance records yet.")
        return pd.DataFrame(columns=["Name", "Date", "Time"])

    df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
    return df


def attendance_summary():
    df = load_all_attendance()
    if df.empty:
        return df

    total_days = df["Date"].nunique()
    summary = df.groupby("Name")["Date"].nunique().reset_index()
    summary.columns = ["Name", "Days Present"]
    summary["Total Days"] = total_days
    summary["Attendance %"] = (summary["Days Present"] / total_days * 100).round(1)
    summary = summary.sort_values("Attendance %", ascending=False).reset_index(drop=True)
    return summary


def day_report(date_str=None):
    """date_str format: 'YYYY-MM-DD'. Defaults to today."""
    if date_str is None:
        date_str = datetime.now().strftime("%Y-%m-%d")
    path = os.path.join(ATTENDANCE_DIR, f"attendance_{date_str}.csv")
    if not os.path.exists(path):
        print(f"No records for {date_str}")
        return pd.DataFrame()
    return pd.read_csv(path)

# Example usage:
# load_all_attendance()
# attendance_summary()
# day_report()


## Typical Workflow

```python
# One-time / whenever adding a new person:
capture_faces("Faria")
capture_faces("Noor")
capture_faces("Asfa")


# After adding people (or new photos):
recognizer, label_map = train_recognizer()

# Each day / class session:
run_attendance_system()

# Anytime, to check reports:
attendance_summary()
day_report("2026-07-20")
```

### Notes / next steps
- `cv2.face` comes from `opencv-contrib-python` — install with:
  `pip install opencv-contrib-python numpy pandas`
- If accuracy is low, capture more samples per person and/or lower `CONFIDENCE_THRESHOLD`.
- To turn this into a website later, you could wrap steps 2-5 as Flask/Django routes and stream the webcam feed via a browser (getUserMedia + a backend endpoint), reusing all the same NumPy/OpenCV/Pandas logic here.
